```bash
pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[OptimalTransport]'
```

- Introduction (/3)
- Preparation et analyse des données (/3)
- Application des méthodes de pre processing (/5)
- Application des méthodes de post processing (/5)
- Analyse, compréhension (/3)
- Conclusion (/1)

## Introduction

nous disposons désormais des images elles-mêmes, ce qui nous permet d'entraîner un véritable modèle de prédiction. L’objectif est de construire un pipeline complet comportant :
- un prétraitement visant à atténuer les biais avant l'entraînement (par exemple via l'algorithme LFR),
- un modèle de classification basé sur les images pour prédire les maladies,
- un post-traitement appliqué aux prédictions pour corriger d’éventuelles inégalités restantes (via l'algorithme Equalized Odds Postprocessing par exemple).

Ce rapport présente la mise en œuvre de ce pipeline, les défis rencontrés, et une évaluation de l'impact des différentes étapes sur la qualité et l’équité des prédictions.

In [1]:
import utils
import os
from constants import *
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix
from aif360.datasets import BinaryLabelDataset
from train_classifieur import train_classifier, pred_classifier






utils.load_env_file()
data_dir = os.getenv("DATA_DIR", "data/default/")
og_metadata_filename="original_metadata.csv"
og_metadata_path = data_dir + og_metadata_filename
pred_output_dir="./expe_log/selected_data/"
print("Travaille sur : ", data_dir)
print("Output en : ", pred_output_dir)
print(og_metadata_path)

Travaille sur :  ../LALAOUI_RAYAN/selected_data/
Output en :  ./expe_log/selected_data/
../LALAOUI_RAYAN/selected_data/original_metadata.csv


In [2]:
# variables et fonctions importante 

map_genre = {"M": 0, "F": 1}
map_viewposition = {"AP": 0, "PA": 1}
map_pred = {"sain": 0, "malade": 1}

fav_lbl = map_pred["sain"]
unfav_lbl = map_pred["malade"]
protected_attributes = ['Patient Gender', '+40ans']

protected_attribute = protected_attributes[1]

priviliged_group = 0
unpriviliged_group = 1


unprivileged_groups = [{protected_attribute: unpriviliged_group}]
privileged_groups = [{protected_attribute: priviliged_group}]


In [3]:

def convert_to_all_numerical(df):
    # Define paths to the train repository
    train_sain_path = data_dir+"/train/sain"
    train_malade_path = data_dir+"/train/malade"

    # Get the list of image filenames in the train repository
    train_images = set(os.listdir(train_sain_path) + os.listdir(train_malade_path))

    df.columns = df.columns.str.strip()
    if "in_train" not in df.columns:
        df["in_train"] = df["Image Index"].apply(lambda x: 1 if x in train_images else 0)
    if 'Finding Labels' in df.columns:
        df_ohe = df['Finding Labels'].str.get_dummies(sep='|').astype(bool)
        df = df.drop(columns=['Finding Labels']).join(df_ohe)
    if "preds" in df.columns and not pd.api.types.is_numeric_dtype(df["preds"]):
        df["preds"] = df["preds"].map({"sain": 0, "malade": 1})
    if "labels" in df.columns and not pd.api.types.is_numeric_dtype(df["labels"]):
        df["labels"] = df["labels"].map({"sain": 0, "malade": 1})
    if not pd.api.types.is_numeric_dtype(df["Patient Gender"]):
        df["Patient Gender"] = df["Patient Gender"].map(map_genre)
    if "View Position" in df.columns and not pd.api.types.is_numeric_dtype(df["View Position"]):
        df["View Position"] = df["View Position"].map(map_viewposition)
    if "+40ans" not in df.columns:
        df["+40ans"] = (df["Patient Age"] > 40).astype(int) 
    return df

In [4]:

from aif360.sklearn.metrics import *


def get_group_metrics(
    y_true,
    y_pred=None,
    prot_attr=None,
    priv_group=1,
    pos_label=1,
    sample_weight=None,
):
    group_metrics = {}
    group_metrics["base rate"] = base_rate(
        y_true=y_true, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["SPD"] = statistical_parity_difference(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["DI"] = disparate_impact_ratio(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    if not y_pred is None:
        group_metrics["equal_opportunity_difference"] = equal_opportunity_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["average_odds_difference"] = average_odds_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["conditional_demographic_disparity"] = conditional_demographic_disparity(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["smoothed_edf"] = smoothed_edf(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["df_bias_amplification"] = df_bias_amplification(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
    return group_metrics


pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[AdversarialDebiasing]'
/home/rayan/Documents/Fairness/projet/myenv/lib/python3.11/site-packages/inFairness/utils/ndcg.py:37: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  vect_normalized_discounted_cumulative_gain = vmap(
/home/rayan/Documents/Fairness/projet/myenv/lib/python3.11/site-packages/inFairness/utils/ndcg.py:48: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `

In [5]:
def train_and_predict(metadata_csv, outputcsv, force_training=False):
    os.makedirs(pred_output_dir, exist_ok=True)
    csv_out = os.path.join(pred_output_dir, outputcsv)
    csv_in=data_dir+metadata_csv
    print(csv_in)
    if force_training or not os.path.exists(csv_out):
        print("Entrainement du classifieur...")
        ckpt_path, ckpt_score = train_classifier(
            logdir=pred_output_dir,
            datadir=data_dir,
            csv=csv_in,
        )
        print("Génerations des predictions...")
        pred_classifier(
            datadir=data_dir,
            csv_in=csv_in,
            csv_out=csv_out,
            ckpt_path=ckpt_path
        )
    else:
        print(f"Les prédiction existent déjà à {csv_out} -- abandon de l'entraînement")

def intoBinaryLabelDataset(df):
    manquantes = [attribute for attribute in protected_attributes if attribute not in df.columns]

    if manquantes:
        raise ValueError(f"Les colonnes protégées suivantes sont manquantes dans le dataset : {', '.join(manquantes)}")

    dataset = BinaryLabelDataset(
        favorable_label=fav_lbl,  # "Sain" est la classe favorable
        unfavorable_label=unfav_lbl,  # "Malade" est la classe défavorable
        df=df,
        label_names=["labels"],
        protected_attribute_names=protected_attributes
    )
    return dataset

def getMetric(df, prot_attr):
    if isinstance(prot_attr, list) :
        raise RuntimeError("On ne peut pas faire de metriquesurplusieur attr protegé")
    df = convert_to_all_numerical(df)
    preds = df["preds"]
    labels= df["labels"]
    weights = df["WEIGHTS"]

    metrics_after_reweight = get_group_metrics(
        y_true=labels,
        y_pred=preds,
        prot_attr=df[prot_attr],
        priv_group=1,
        pos_label=1,
        sample_weight=weights
    )
    return metrics_after_reweight

    


## Préparations des données

Notamment pour les converitir dans un ``BinaryLabelDataset``

In [6]:
df = pd.read_csv(og_metadata_path)

print(df.columns)
imageid_df = df.copy()[["Image Index", patientid]]
original_df = df.copy()
df = convert_to_all_numerical(df)

df.head() # Y'a toujours Image index !!

Index(['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID',
       'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width',
       'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'WEIGHTS'],
      dtype='object')


,Image Index,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],...,Fibrosis,Hernia,Infiltration,Mass,No Finding,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax,+40ans
0,00000001_000.png,0,1,58,0,1,2682,2749,0.143,0.143,...,False,False,False,False,False,False,False,False,False,1
1,00000005_006.png,6,5,70,1,1,2992,2991,0.143,0.143,...,False,False,True,False,False,False,False,False,False,1
2,00000014_000.png,0,14,61,1,1,2048,2500,0.171,0.171,...,False,False,False,False,True,False,False,False,False,1
3,00000048_000.png,0,48,46,1,1,2834,2641,0.143,0.143,...,False,False,False,False,True,False,False,False,False,1
4,00000051_000.png,0,51,55,0,1,3056,2544,0.139,0.139,...,False,False,True,False,False,False,False,False,False,1


In [7]:
# recupperer les predictions sans aucun changement
train_and_predict(og_metadata_filename, "original_preds.csv")

../LALAOUI_RAYAN/selected_data/original_metadata.csv
Les prédiction existent déjà à ./expe_log/selected_data/original_preds.csv -- abandon de l'entraînement


In [8]:
preddf = pd.read_csv(pred_output_dir+"original_preds.csv")
preddf = convert_to_all_numerical(preddf)

metrics_before_training = getMetric(preddf, protected_attribute)

def compare_to_base_preds(metrics_after):
    print(f"{'Avant':^10} {'→':^7} {'Après':^10} | {'Différence':>10} | {'Métrique'}")
    print("-" * 55)
    for metric in metrics_before_training.keys():
        before = metrics_before_training[metric]
        after = metrics_after[metric]
        change = after - before
        print(f"{before:^10.4f} {'→':^7} {after:^10.4f} | {change:>10.4f} | {metric}")


In [9]:
allmetrics = {}
allmetrics['Without proc'] = metrics_before_training

## Analyse

On va surtout s'interesser à l'âge 

In [10]:
utils.plot_age_dist(original_df)

In [11]:

def plot_confusion_matrices_side_by_side(df, group_column, labels=["sain", "malade"], normalize=False):
    y_true = df["labels"].values
    y_pred = df["preds"].values
    unique_groups = df[group_column].unique()
    n_groups = len(unique_groups)
    
    fig = make_subplots(
        rows=1,
        cols=n_groups,
        subplot_titles=[f"{group_column} = {val}" for val in unique_groups],
        horizontal_spacing=0.25 
    )

    for i, group_value in enumerate(unique_groups):
        group_df = df[df[group_column] == group_value]
        y_true_group = y_true[group_df.index]
        y_pred_group = y_pred[group_df.index]

        cm = confusion_matrix(y_true_group, y_pred_group)

        if normalize:
            cm = cm.astype('float') / len(y_true_group) * 100
        z_text = [[f"{val:.2f}%" if normalize else str(int(val)) for val in row] for row in cm]
        fig.add_trace(
            go.Heatmap(
                z=cm, x=labels, y=labels,
                colorscale="Blues",
                showscale=False,
                zmin=0, zmax=100 if normalize else None,
                text=z_text, texttemplate="%{text}", hoverinfo="z"
            ),
            row=1, col=i+1
        )

        fig.update_xaxes(title_text="Prédiction", row=1, col=i+1)
        fig.update_yaxes(title_text="Vérité", row=1, col=i+1, autorange="reversed")

    fig.update_layout(
        title_text="Matrices de Confusion par Groupe",
        height=400,
        width=420 * n_groups  # plus de largeur pour espacer
    )
    
    fig.show()


In [12]:
preddf['+40ans'] = preddf['Patient Age'] >= 40
plot_confusion_matrices_side_by_side(
    df=preddf,
    group_column='+40ans',
    labels=["sain", "malade"],
)

In [13]:


error_rate_df = pd.DataFrame(columns=['method', 'global', '+40ans', '-40ans', 'M', 'F'])

def compute_error_rate(y_true, y_pred, group_name="inconnu"):
    if len(y_true) == 0:
        warnings.warn(f"[{group_name}] Aucun exemple → erreur mise à 0.")
        return 0.0

    cm = confusion_matrix(y_true, y_pred)
    total = cm.sum()
    correct = np.trace(cm)

    if total == 0:
        warnings.warn(f"[{group_name}] Matrice de confusion vide → erreur mise à 0.")
        return 0.0

    return (total - correct) / total * 100

def add_error_rate(df, method):
    global error_rate_df
    df = df.copy()
    # Calcul global
    global_error = compute_error_rate(df["labels"], df["preds"], group_name="global")

    df_plus_40 = df[df['+40ans'] == 1]
    df_moins_40 = df[df['+40ans'] == 0]
    df_hommes = df[df['Patient Gender'] == 1]
    df_femmes = df[df['Patient Gender'] == 0]

    error_plus_40 = compute_error_rate(df_plus_40["labels"], df_plus_40["preds"], "+40 ans")
    error_moins_40 = compute_error_rate(df_moins_40["labels"], df_moins_40["preds"], "-40 ans")
    error_hommes = compute_error_rate(df_hommes["labels"], df_hommes["preds"], "Hommes")
    error_femmes = compute_error_rate(df_femmes["labels"], df_femmes["preds"], "Femmes")

    new_row = {
        'method': method,
        'global': global_error,
        '+40ans': error_plus_40,
        '-40ans': error_moins_40,
        'M': error_hommes,
        'F': error_femmes
    }

    error_rate_df.loc[len(error_rate_df)] = new_row


In [14]:
add_error_rate(preddf, 'Normal')
error_rate_df

,method,global,+40ans,-40ans,M,F
0,Normal,30.0,33.604888,23.166023,29.369628,30.548628


## pre processing

#### Pre pre processing

In [15]:

og_preddf = pd.read_csv(pred_output_dir+"original_preds.csv")
og_preddf = convert_to_all_numerical(og_preddf)




filtered_df = og_preddf.drop(["View Position", "Finding Labels", "Image Index"], axis=1, errors="ignore")
train_df = filtered_df[filtered_df["in_train"]==1].copy().reset_index()
test_df = filtered_df[filtered_df["in_train"]==0].copy().reset_index()

dataset = intoBinaryLabelDataset(filtered_df)
train_dataset = intoBinaryLabelDataset(train_df)
test_dataset = intoBinaryLabelDataset(test_df)


a=len(train_dataset.instance_weights)
b=len(test_dataset.instance_weights)
c=len(dataset.instance_weights)
assert(a+b==c)
# train_df

print(f"taille du train : {len(train_df)}")
print(f"taille du test : {len(test_df)}")

taille du train : 1125
taille du test : 375


#### reweight

In [16]:
sensitive_attr = "+40ans"
unprivileged_groups, privileged_groups=[{sensitive_attr: 0}], [{sensitive_attr: 1}]

In [17]:
from aif360.algorithms.preprocessing import Reweighing

rw = Reweighing(unprivileged_groups, privileged_groups)
rw.fit(train_dataset)
transformed_dataset = rw.transform(dataset)

csv_df = original_df.copy()
csv_df["WEIGHTS"] = transformed_dataset.instance_weights
csv_df.to_csv(data_dir+"/"+"reweighted_metadata.csv", index=False)


In [18]:
train_and_predict("reweighted_metadata.csv", "reweighted_preds.csv")

../LALAOUI_RAYAN/selected_data/reweighted_metadata.csv
Les prédiction existent déjà à ./expe_log/selected_data/reweighted_preds.csv -- abandon de l'entraînement


In [19]:
rw_pred = pd.read_csv(pred_output_dir+"reweighted_preds.csv")
rw_pred = convert_to_all_numerical(rw_pred)
metrics_after_reweight = getMetric(rw_pred, sensitive_attr)
allmetrics["Reweight"] = metrics_after_reweight

In [20]:
compare_to_base_preds(metrics_after_reweight)

  Avant       →      Après    | Différence | Métrique
-------------------------------------------------------
  0.4595      →      0.4595   |     0.0000 | base rate
 -0.0473      →     -0.0212   |     0.0261 | SPD
  0.8978      →      0.9266   |     0.0288 | DI
  0.0593      →      0.0872   |     0.0279 | equal_opportunity_difference
 -0.0260      →      0.0002   |     0.0261 | average_odds_difference
 -0.0120      →     -0.0066   |     0.0054 | conditional_demographic_disparity
  0.1076      →      0.0754   |    -0.0321 | smoothed_edf
  0.0351      →      0.0030   |    -0.0321 | df_bias_amplification


In [21]:
add_error_rate(rw_pred, "Reweight")
error_rate_df

,method,global,+40ans,-40ans,M,F
0,Normal,30.000000,33.604888,23.166023,29.369628,30.548628
1,Reweight,29.133333,34.243697,20.255474,27.077364,30.922693


In [22]:
plot_confusion_matrices_side_by_side(
    df=rw_pred,
    group_column='+40ans',
    labels=["sain", "malade"],
)
# plot_confusion_matrix_by_group(rw_pred["labels"], rw_pred["preds"], rw_pred, group_columns=["+40ans"], labels=[0, 1])

#### DIR

In [23]:
from aif360.algorithms.preprocessing import DisparateImpactRemover

def apply_disparate_impact_remover(original_df, repair_level=1.0):
    label_col = "labels"

    protected_attr = "+40ans"

    # Colonnes à garder pour la réparation
    dir_features = ["Patient Age", "Patient Gender", protected_attr, label_col, "WEIGHTS"] 
    
    df_dir = original_df[dir_features].copy()
    dataset = BinaryLabelDataset(
        df=df_dir,
        label_names=[label_col],
        protected_attribute_names=[protected_attr]
    )

    # pour les restaurer ensuite
    patient_ids = original_df["Patient ID"].astype(str).tolist()
    dataset.instance_names = [[pid] for pid in patient_ids]

    dir = DisparateImpactRemover(sensitive_attribute=protected_attr, repair_level=repair_level)
    repaired_dataset = dir.fit_transform(dataset)
    repaired_df = pd.DataFrame(
        data=repaired_dataset.features,
        columns=repaired_dataset.feature_names
    )
    repaired_df[label_col] = repaired_dataset.labels
    # on remet les ids et les images
    repaired_df["Patient ID"] = [int(pid[0]) for pid in repaired_dataset.instance_names]
    imageid_df["Patient ID"] = imageid_df["Patient ID"].astype(int)
    repaired_df = repaired_df.merge(imageid_df, on="Patient ID", how="left")

    columns_to_add = ["in_train"]
    for col in columns_to_add:
        repaired_df[col] = original_df[col].values

    repaired_df["+40ans"] = (repaired_df["Patient Age"] >= 40).astype(int)
    return repaired_df


In [24]:
# pournepas utiliser d'info du datatest dans le train -> data leakage

repaired_train_df = apply_disparate_impact_remover(train_df)
repaired_test_df =  apply_disparate_impact_remover(test_df)

repaired_df = pd.concat([repaired_train_df, repaired_test_df], ignore_index=True)

repaired_df.to_csv(data_dir+"/"+"dir_metadata.csv", index=False)


In [25]:
train_and_predict("dir_metadata.csv", "dir_preds.csv")

../LALAOUI_RAYAN/selected_data/dir_metadata.csv
Les prédiction existent déjà à ./expe_log/selected_data/dir_preds.csv -- abandon de l'entraînement


In [26]:
dir_pred = pd.read_csv(pred_output_dir+"dir_preds.csv")
dir_pred = convert_to_all_numerical(dir_pred)
metrics_after_dir = getMetric(dir_pred, sensitive_attr)
compare_to_base_preds(metrics_after_dir)
allmetrics["DIR"] = metrics_after_dir

  Avant       →      Après    | Différence | Métrique
-------------------------------------------------------
  0.4595      →      0.4484   |    -0.0111 | base rate
 -0.0473      →     -0.1302   |    -0.0829 | SPD
  0.8978      →      0.7727   |    -0.1250 | DI
  0.0593      →     -0.0538   |    -0.1131 | equal_opportunity_difference
 -0.0260      →     -0.1247   |    -0.0988 | average_odds_difference
 -0.0120      →      0.0103   |     0.0223 | conditional_demographic_disparity
  0.1076      →      0.2605   |     0.1529 | smoothed_edf
  0.0351      →      0.2523   |     0.2172 | df_bias_amplification


In [27]:
add_error_rate(dir_pred, "Dir")
error_rate_df

,method,global,+40ans,-40ans,M,F
0,Normal,30.000000,33.604888,23.166023,29.369628,30.548628
1,Reweight,29.133333,34.243697,20.255474,27.077364,30.922693
2,Dir,25.800000,34.375000,25.613079,27.220630,24.563591


In [28]:
plot_confusion_matrices_side_by_side(
    df=dir_pred,
    group_column='+40ans',
    labels=["sain", "malade"],
)
# plot_confusion_matrix_by_group(dir_df["labels"], dir_df["preds"], dir_df, group_columns=["+40ans"], labels=[0, 1])

#### LFR

In [29]:
from aif360.algorithms.preprocessing import LFR

def apply_lfr(df, maxiter=5000, maxfun=5000):

    # colonnes à garder pour la réparation
    dir_features = ["Patient Age", "Patient Gender", "+40ans", "labels", "WEIGHTS"] 
    df_dir = df[dir_features].copy()
    
    dataset = intoBinaryLabelDataset(df_dir)

    # stockage des Patient ID pour les restaurer ensuite
    patient_ids = df["Patient ID"].astype(str).tolist()
    dataset.instance_names = [[pid] for pid in patient_ids]

    TR = LFR(
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups,
        k=5,
        Ax=0.001, Ay=0.1, Az=1.0,
        print_interval=500,
        verbose=1,
        seed=None
    )

    TR = TR.fit(dataset, maxiter=maxiter, maxfun=maxfun)
    repaired_dataset = TR.transform(dataset)
    repaired_df = pd.DataFrame(
        data=repaired_dataset.features,
        columns=repaired_dataset.feature_names
    )
    repaired_df["labels"] = repaired_dataset.labels

    # Réinsertion des Patient ID, du train et des images
    repaired_df["Patient ID"] = [int(pid[0]) for pid in repaired_dataset.instance_names]
    imageid_df["Patient ID"] = imageid_df["Patient ID"].astype(int)
    repaired_df = repaired_df.merge(imageid_df, on="Patient ID", how="left")
    repaired_df["in_train"] = df["in_train"].values

    repaired_df["+40ans"] = (repaired_df["Patient Age"] >= 40).astype(int)
    repaired_df.to_csv(data_dir + "lfr_metadata.csv", index=False)

    return repaired_df

In [30]:
truc = apply_lfr(train_df)
truc2 = apply_lfr(test_df)
repaired_df = pd.concat([truc, truc2], ignore_index=True)
repaired_df.to_csv(data_dir+"lfr_metadata.csv", index=False)


step: 0, loss: 1.062582738933817, L_x: 985.1961939042387,  L_y: 0.7658065035407556,  L_z: 0.0008058946755024332
step: 500, loss: 0.460816548746636, L_x: 325.3355272013323,  L_y: 0.7779958090243231,  L_z: 0.05768144064287138
step: 1000, loss: 0.2769557282905868, L_x: 122.43709189412479,  L_y: 0.688973518330842,  L_z: 0.08562128456337781
step: 1500, loss: 0.2173445304670238, L_x: 146.03129327501247,  L_y: 0.6907290003481894,  L_z: 0.0022403371571923843
step: 2000, loss: 0.21310957309184653, L_x: 142.83859027845097,  L_y: 0.6878388891428686,  L_z: 0.0014870938991086834
step: 0, loss: 1.1068101220889592, L_x: 1020.7241744901971,  L_y: 0.8464673471361819,  L_z: 0.0014392128851439645
step: 500, loss: 0.3280788426017867, L_x: 133.98946842414026,  L_y: 0.7000401373867483,  L_z: 0.12408536043897163
step: 1000, loss: 0.24218032042215368, L_x: 158.05418563865052,  L_y: 0.8412613464669151,  L_z: 1.3681165091910907e-10


In [31]:
train_and_predict("lfr_metadata.csv", "lfr_preds.csv")

../LALAOUI_RAYAN/selected_data/lfr_metadata.csv
Les prédiction existent déjà à ./expe_log/selected_data/lfr_preds.csv -- abandon de l'entraînement


In [32]:
lfr_pred = pd.read_csv(pred_output_dir+"lfr_preds.csv")
lfr_pred = convert_to_all_numerical(lfr_pred)
metrics_after_lfr = getMetric(lfr_pred, sensitive_attr)

allmetrics["LFR"] = metrics_after_lfr
compare_to_base_preds(metrics_after_lfr)

  Avant       →      Après    | Différence | Métrique
-------------------------------------------------------
  0.4595      →      0.4604   |     0.0009 | base rate
 -0.0473      →     -0.0124   |     0.0349 | SPD
  0.8978      →      0.9480   |     0.0502 | DI
  0.0593      →     -0.0182   |    -0.0775 | equal_opportunity_difference
 -0.0260      →     -0.0137   |     0.0123 | average_odds_difference
 -0.0120      →     -0.0043   |     0.0077 | conditional_demographic_disparity
  0.1076      →      0.0523   |    -0.0552 | smoothed_edf
  0.0351      →      0.0461   |     0.0111 | df_bias_amplification


In [33]:
add_error_rate(lfr_pred, "Lfr")
error_rate_df

/tmp/ipykernel_4432/3362370423.py:5: UserWarning:

[Hommes] Aucun exemple → erreur mise à 0.

/tmp/ipykernel_4432/3362370423.py:5: UserWarning:

[Femmes] Aucun exemple → erreur mise à 0.



,method,global,+40ans,-40ans,M,F
0,Normal,30.000000,33.604888,23.166023,29.369628,30.548628
1,Reweight,29.133333,34.243697,20.255474,27.077364,30.922693
2,Dir,25.800000,34.375000,25.613079,27.220630,24.563591
3,Lfr,31.466667,31.302521,31.751825,0.000000,0.000000


In [34]:
plot_confusion_matrices_side_by_side(
    df=lfr_pred,
    group_column='+40ans',
    labels=["sain", "malade"],
)

## Post processing

In [49]:
from aif360.algorithms.postprocessing.reject_option_classification import RejectOptionClassification

metric_name = "Statistical parity difference"


def apply_ROC_to_preds(test_df, low_class_thresh = 0.01, high_class_thresh = 0.99, metric_ub = 0.05, metric_lb = -0.05):
    # -------- 1. Préparation des données -------- 
    test_df = test_df.drop(columns=["Image Index"])
    train_predictions = test_df[test_df['in_train'] == 0]
    test_dataset = intoBinaryLabelDataset(train_predictions)

    test_with_preds : BinaryLabelDataset = test_dataset.copy(deepcopy=True)
    test_with_preds.labels = train_predictions["preds"].values.reshape(-1, 1)
    test_with_preds.scores = train_predictions["logits_1"].values.reshape(-1, 1)

    # -------- 2. Application du Reject Option Classification --------
    ROC = RejectOptionClassification(
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups,
        low_class_thresh=low_class_thresh,
        high_class_thresh=high_class_thresh,
        num_class_thresh=100,
        num_ROC_margin=50,
        metric_name=metric_name,
        metric_ub=metric_ub,
        metric_lb=metric_lb
    ).fit(test_dataset, test_with_preds)

    # -------- 3. Prédictions corrigées par ROC --------
    transformed = ROC.predict(test_with_preds)

    newcols = list(test_df.columns)
    newcols.remove("labels")
    transformed_df = pd.DataFrame(transformed.features, columns=newcols)
    transformed_df['labels'] = train_predictions['labels'].values
    transformed_df['preds'] = transformed.labels.reshape(-1)
    transformed_df['logits_1'] = transformed.scores.reshape(-1)
    transformed_df['in_train'] = train_predictions['in_train'].values 
    
    return transformed_df



In [42]:
def cross_validate_ROC(test_df, metrics_to_try=None, class_thresholds=None, fairness_bounds=None, weights=False):
    test_df = test_df[test_df["in_train"]==0]
    test_dataset = BinaryLabelDataset(
        favorable_label=0,  
        unfavorable_label=1,  
        df=convert_to_all_numerical(test_df).select_dtypes(include=['int64', 'float64']),
        label_names=["labels"],
        protected_attribute_names=[protected_attribute]
    )

    # Default parameters
    if metrics_to_try is None:
        metrics_to_try = [
            "Statistical parity difference",
            "Equal opportunity difference",
            "Average odds difference"
        ]
    if class_thresholds is None:
        class_thresholds = [(0.01, 0.99), (0.001, 0.999)] 
    if fairness_bounds is None:
        fairness_bounds = [
            (-0.01, 0.01),
            (-0.05, 0.05),
            (-0.1, 0.1),
            (-0.25, 0.25)
        ]

  
    test_with_preds = test_dataset.copy(deepcopy=True)
    test_with_preds.labels = test_df["preds"].values.reshape(-1, 1)
    test_with_preds.scores = test_df["logits_1"].values.reshape(-1, 1)

    results_dfs = {}
    for metric in metrics_to_try:
        metric_results = []
        for low_thresh, high_thresh in class_thresholds:
            for lb, ub in fairness_bounds:
                try:
                   
                    # Initialize ROC with current parameters
                    ROC = RejectOptionClassification(
                        unprivileged_groups=unprivileged_groups,
                        privileged_groups=privileged_groups,
                        low_class_thresh=low_thresh,
                        high_class_thresh=high_thresh,
                        num_class_thresh=100,
                        num_ROC_margin=50,
                        metric_name=metric,
                        metric_ub=ub,
                        metric_lb=lb
                    )

                    
                    
                   
                    
                    ROC = ROC.fit(test_dataset, test_with_preds)
                    
                    transformed_dataset = ROC.predict(test_with_preds)
                    
                    # Calculate metrics using the transformed predictions
                    weight_column = None
                    if weights:
                        if 'WEIGHTS' in test_dataset.feature_names:
                            weight_column = test_dataset.features[:, test_dataset.feature_names.index('WEIGHTS')]
                    
                    metrics = get_group_metrics(
                        y_true=test_dataset.labels[:,0],
                        y_pred=transformed_dataset.labels[:,0],
                        prot_attr=test_dataset.protected_attributes[:, 0],
                        pos_label=1,
                        sample_weight=weight_column
                    )
                    
                    
                    result_row = {
                        'low_class_thresh': low_thresh,
                        'high_class_thresh': high_thresh,
                        'metric_lb': lb,
                        'metric_ub': ub,
                        **metrics
                    }
                    metric_results.append(result_row)
                except Exception as e:
                    print(e)
                    result_row = {
                        'low_class_thresh': low_thresh,
                        'high_class_thresh': high_thresh,
                        'metric_lb': lb,
                        'metric_ub': ub,
                        'distance': np.nan,
                        'error': str(e)
                    }
                    metric_results.append(result_row)
        
        df = pd.DataFrame(metric_results)
        df.set_index(['low_class_thresh', 'high_class_thresh', 'metric_lb', 'metric_ub'], inplace=True)
        results_dfs[metric] = df
    
    return results_dfs
results_dfs_rw = cross_validate_ROC(rw_pred)
results_dfs_dir = cross_validate_ROC(dir_pred)

KeyboardInterrupt: 

In [43]:
def plot_parameter_results(results_df):
    
    df = results_df.reset_index()

    metric_cols = [col for col in df.columns if col not in ['low_class_thresh', 'high_class_thresh', 'metric_lb', 'metric_ub']]
    df['param_combo'] = df.apply(
        lambda x: f'L:{x.low_class_thresh:.3f}, H:{x.high_class_thresh:.3f}\nLB:{x.metric_lb:.3f}, UB:{x.metric_ub:.3f}', 
        axis=1
    )
    
    n_params = len(df['param_combo'].unique())
    n_rows = (n_params + 1) // 2
    n_cols = 2

    fig = make_subplots(
        rows=n_rows, 
        cols=n_cols,
        subplot_titles=df['param_combo'].unique(),
        vertical_spacing=0.2
    )
    
    # Plot metrics for each parameter combination
    for i, param_combo in enumerate(df['param_combo'].unique()):
        row = (i // 2) + 1
        col = (i % 2) + 1
        
        param_data = df[df['param_combo'] == param_combo]
        
        bar = go.Bar(
            x=metric_cols,
            y=param_data[metric_cols].values[0],
            name=param_combo
        )
        fig.add_trace(bar, row=row, col=col)
    
        max_abs_val = max(abs(param_data[metric_cols].values[0]))
        fig.update_yaxes(range=[-max_abs_val*1.1, max_abs_val*1.1], row=row, col=col)
    fig.update_layout(
        height=300 * n_rows,
        width=1200,
        showlegend=False,
        title_text="Parameter Combinations Results (All Metrics)"
    )
    
   
    fig.update_xaxes(tickangle=45)
    
    return fig

fig = plot_parameter_results(results_dfs_rw["Statistical parity difference"])
fig.show()

In [38]:
fig = plot_parameter_results(results_dfs_dir["Statistical parity difference"])
fig.show()

In [50]:

high_class , low_class , lb , ub = 0.99, 0.01,-0.05, 0.05
trans = apply_ROC_to_preds(rw_pred, low_class_thresh=low_class, high_class_thresh=high_class, metric_lb=lb, metric_ub=ub)
metric_roc_reweigth = getMetric(trans, protected_attribute)

high_class , low_class , lb , ub = 0.999, 0.001,-0.010, 0.010
trans = apply_ROC_to_preds(dir_pred, low_class_thresh=low_class, high_class_thresh=high_class, metric_lb=lb, metric_ub=ub)
metric_roc_dir = getMetric(trans, protected_attribute)


allmetrics['ROC+RW'] = metric_roc_reweigth
allmetrics['ROC+DIR'] = metric_roc_dir


print("Via rw")
compare_to_base_preds(metric_roc_reweigth)
print("Via dir")
compare_to_base_preds(metric_roc_dir)


Via rw
  Avant       →      Après    | Différence | Métrique
-------------------------------------------------------
  0.4595      →      0.4783   |     0.0188 | base rate
 -0.0473      →     -0.0267   |     0.0206 | SPD
  0.8978      →      0.7525   |    -0.1452 | DI
  0.0593      →      0.0178   |    -0.0415 | equal_opportunity_difference
 -0.0260      →     -0.0036   |     0.0224 | average_odds_difference
 -0.0120      →     -0.0218   |    -0.0098 | conditional_demographic_disparity
  0.1076      →      0.2588   |     0.1513 | smoothed_edf
  0.0351      →     -0.0373   |    -0.0724 | df_bias_amplification
Via dir
  Avant       →      Après    | Différence | Métrique
-------------------------------------------------------
  0.4595      →      0.4680   |     0.0084 | base rate
 -0.0473      →     -0.0138   |     0.0335 | SPD
  0.8978      →      0.9638   |     0.0661 | DI
  0.0593      →     -0.1456   |    -0.2049 | equal_opportunity_difference
 -0.0260      →     -0.0491   |    -0.02

## CalibratedEqOddsPostprocessing

In [ ]:
from aif360.algorithms.postprocessing.calibrated_eq_odds_postprocessing import CalibratedEqOddsPostprocessing



def apply_CEO(test_df):
    test_df = test_df[test_df['in_train']==0]
    cost_constraint = "fnr" # "fnr", "fpr", "weighted"
    cpp = CalibratedEqOddsPostprocessing(privileged_groups = privileged_groups,
                                        unprivileged_groups = unprivileged_groups,
                                        cost_constraint=cost_constraint,
                                        seed=42)
    
    pred_dataset = test_dataset.copy(deepcopy=True)
    pred_dataset.labels = test_df["preds"].values.reshape(-1, 1)
    pred_dataset.scores = test_df["logits_1"].values.reshape(-1, 1)
   

    cpp = cpp.fit(test_dataset, pred_dataset)
    calibrated_pred = cpp.predict(pred_dataset)
    
    newcols = calibrated_pred.feature_names
    transformed_df = pd.DataFrame(calibrated_pred.features, columns=newcols)
    transformed_df['labels'] = test_df['labels'].values
    transformed_df['preds'] = calibrated_pred.labels.reshape(-1)
    transformed_df['logits_1'] = calibrated_pred.scores.reshape(-1)
    transformed_df['in_train'] = test_df['in_train'].values

    return transformed_df
    # return m

In [ ]:

trans = apply_CEO(rw_pred)
metric_ceo_reweigth = getMetric(trans, protected_attribute)
trans = apply_CEO(dir_pred)
metric_ceo_dir = getMetric(trans, protected_attribute)


allmetrics['CEO+RW'] = metric_roc_reweigth
allmetrics['CEO+DIR'] = metric_roc_dir


print("Via rw") 
compare_to_base_preds(metric_ceo_reweigth)
print("Via dir")
compare_to_base_preds(metric_ceo_dir)




Via rw
  Avant       →      Après    | Différence | Métrique
-------------------------------------------------------
  0.4595      →      0.4783   |     0.0188 | base rate
 -0.0473      →     -0.2996   |    -0.2523 | SPD
  0.8978      →      0.0000   |    -0.8978 | DI
  0.0593      →     -0.3885   |    -0.4478 | equal_opportunity_difference
 -0.0260      →     -0.2952   |    -0.2692 | average_odds_difference
 -0.0120      →     -0.1369   |    -0.1249 | conditional_demographic_disparity
  0.1076      →      3.6337   |     3.5262 | smoothed_edf
  0.0351      →      3.3376   |     3.3025 | df_bias_amplification
Via dir
  Avant       →      Après    | Différence | Métrique
-------------------------------------------------------
  0.4595      →      0.4783   |     0.0188 | base rate
 -0.0473      →     -0.0832   |    -0.0359 | SPD
  0.8978      →      0.8266   |    -0.0712 | DI
  0.0593      →      0.0125   |    -0.0468 | equal_opportunity_difference
 -0.0260      →     -0.0419   |    -0.01

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


methods = list(allmetrics.keys())
metrics = list(allmetrics[methods[0]].keys())

plots_per_row = 4
n_rows = (len(metrics) + 1) // plots_per_row

fig = make_subplots(
    rows=n_rows, cols=plots_per_row,
    subplot_titles=metrics,
    vertical_spacing=0.15,
    horizontal_spacing=0.1
)

for idx, metric in enumerate(metrics):
    row = idx // plots_per_row + 1
    col = idx % plots_per_row + 1
    y_vals = [allmetrics[method][metric] for method in methods]
    fig.add_trace(
        go.Bar(
            x=methods, y=y_vals,
            name=metric,
            text=[f"{val:.3f}" for val in y_vals],
            textposition='auto'
        ),
        row=row, col=col
    )

fig.update_layout(
    height=350 * n_rows,
    showlegend=False,
    title_text="Comparaison des Métriques par Méthode (2 par ligne)",
    template="plotly_white"
)

fig.show()


In [ ]:
error_rate_df

,method,global,+40ans,-40ans,M,F
0,Normal,30.000000,33.604888,23.166023,29.369628,30.548628
1,Reweight,29.133333,34.243697,20.255474,27.077364,30.922693
2,Dir,25.800000,34.375000,25.613079,27.220630,24.563591
3,Lfr,31.466667,31.302521,31.751825,0.000000,0.000000


## Conclusion